# WiFi CSI Localization v2 — Advanced Models
**Baseline**: ResNet 2.84m, LightGBM 3.21m (3D distance error)

**Goal**: Beat baseline with deeper architectures, attention, and better training.

**Experiments**:
1. Multi-Scale CNN (parallel conv branches)
2. Transformer encoder on subcarrier sequence
3. Deep ResNet + Squeeze-and-Excitation attention
4. Ensemble of all three

**Runtime**: L4 / H100 GPU

In [ ]:
!pip install -q lightgbm pyarrow

import numpy as np
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import time, gc, os, json, shutil, math

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}, VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
# Mount Drive and load data
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

SAVE_DIR = '/content/drive/MyDrive/fyp'
os.makedirs(SAVE_DIR, exist_ok=True)

DRIVE_PATH = None
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    if 'haloc_compact.parquet' in files:
        DRIVE_PATH = os.path.join(root, 'haloc_compact.parquet')
        break
print(f'Drive: {DRIVE_PATH}')

LOCAL_PATH = '/content/haloc_compact.parquet'
if not os.path.exists(LOCAL_PATH):
    shutil.copy(DRIVE_PATH, LOCAL_PATH)
print(f'Local: {os.path.getsize(LOCAL_PATH)/1e6:.1f} MB')

In [ ]:
# Load data
N_SUB = 128
amp_cols = [f'amp_{i}' for i in range(N_SUB)]
phase_cols = [f'phase_{i}' for i in range(N_SUB)]
table = pq.read_table(LOCAL_PATH, columns=amp_cols + phase_cols + ['x', 'y', 'z', 'split'])

X = np.empty((len(table), N_SUB * 2), dtype=np.float32)
for i, col in enumerate(amp_cols + phase_cols):
    X[:, i] = table.column(col).to_numpy().astype(np.float32)

y_x = table.column('x').to_numpy().astype(np.float32)
y_y = table.column('y').to_numpy().astype(np.float32)
y_z = table.column('z').to_numpy().astype(np.float32)
Y = np.column_stack([y_x, y_y, y_z])

split = table.column('split').to_pylist()
idx_train = np.array([i for i, s in enumerate(split) if s == 'train'])
idx_val = np.array([i for i, s in enumerate(split) if s == 'val'])
idx_test = np.array([i for i, s in enumerate(split) if s == 'test'])
del table; gc.collect()

# Normalize features
X_mean = X[idx_train].mean(0)
X_std = X[idx_train].std(0) + 1e-8
X = (X - X_mean) / X_std

# Normalize targets (helps training converge)
Y_mean = Y[idx_train].mean(0)
Y_std = Y[idx_train].std(0) + 1e-8
Y_norm = (Y - Y_mean) / Y_std

print(f'Loaded: {X.shape[0]:,} rows, {X.shape[1]} features')
print(f'Train: {len(idx_train):,} | Val: {len(idx_val):,} | Test: {len(idx_test):,}')
print(f'Target means: x={Y_mean[0]:.2f}, y={Y_mean[1]:.2f}, z={Y_mean[2]:.2f}')
print(f'Target stds:  x={Y_std[0]:.2f}, y={Y_std[1]:.2f}, z={Y_std[2]:.2f}')

In [ ]:
# Dataloaders with data augmentation
class CSIDataset(torch.utils.data.Dataset):
    def __init__(self, X, Y, augment=False):
        self.X = torch.tensor(X)
        self.Y = torch.tensor(Y)
        self.augment = augment

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx]  # (256,)
        y = self.Y[idx]
        # Reshape to 2-channel: (2, 128)
        x = x.view(2, N_SUB)
        if self.augment:
            # Gaussian noise
            x = x + torch.randn_like(x) * 0.05
            # Random subcarrier masking (drop 5% of subcarriers)
            mask = torch.rand(N_SUB) > 0.05
            x = x * mask.unsqueeze(0)
        return x, y

train_ds = CSIDataset(X[idx_train], Y_norm[idx_train], augment=True)
val_ds = CSIDataset(X[idx_val], Y_norm[idx_val], augment=False)
test_ds = CSIDataset(X[idx_test], Y_norm[idx_test], augment=False)

BS = 2048
train_loader = DataLoader(train_ds, batch_size=BS, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BS*2, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BS*2, num_workers=2, pin_memory=True)
print(f'Loaders ready. Train batches: {len(train_loader)}')

In [ ]:
# ==========================================
#  Shared training function
# ==========================================
def train_model(model, name, epochs=120, lr=1e-3, max_lr=3e-3):
    criterion = nn.HuberLoss(delta=1.0)  # More robust than MSE
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.OneCycleLR(optimizer, max_lr=max_lr, epochs=epochs, steps_per_epoch=len(train_loader))
    scaler = torch.amp.GradScaler()

    hist = {'train_loss': [], 'val_loss': [], 'val_mae': []}
    best_val_mae = float('inf')
    save_path = f'{SAVE_DIR}/best_{name}.pt'

    for epoch in range(epochs):
        t0 = time.time()
        model.train()
        tloss = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            with torch.amp.autocast(device_type='cuda'):
                loss = criterion(model(xb), yb)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            tloss += loss.item() * len(xb)
        tloss /= len(train_loader.dataset)

        model.eval()
        vloss, all_p, all_l = 0, [], []
        with torch.no_grad(), torch.amp.autocast(device_type='cuda'):
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                out = model(xb)
                vloss += criterion(out, yb).item() * len(xb)
                # Denormalize for MAE in meters
                pred_m = out.cpu().numpy() * Y_std + Y_mean
                true_m = yb.cpu().numpy() * Y_std + Y_mean
                all_p.append(pred_m); all_l.append(true_m)
        vloss /= len(val_loader.dataset)
        vp, vl = np.concatenate(all_p), np.concatenate(all_l)
        vmae_3d = np.sqrt(((vp - vl)**2).sum(axis=1)).mean()

        hist['train_loss'].append(tloss)
        hist['val_loss'].append(vloss)
        hist['val_mae'].append(vmae_3d)

        tag = ''
        if vmae_3d < best_val_mae:
            best_val_mae = vmae_3d
            torch.save(model.state_dict(), save_path)
            tag = ' *best*'

        if (epoch+1) % 10 == 0 or tag:
            print(f'  [{name}] Ep {epoch+1:3d}/{epochs} | TL: {tloss:.4f} | VL: {vloss:.4f} | 3D: {vmae_3d:.3f}m | {time.time()-t0:.1f}s{tag}')

    print(f'  [{name}] Best val 3D MAE: {best_val_mae:.4f}m\n')
    return hist, best_val_mae


def eval_model(model, name):
    model.eval()
    preds, labels = [], []
    with torch.no_grad(), torch.amp.autocast(device_type='cuda'):
        for xb, yb in test_loader:
            out = model(xb.to(device)).cpu().numpy()
            preds.append(out * Y_std + Y_mean)
            labels.append(yb.numpy() * Y_std + Y_mean)
    p, l = np.concatenate(preds), np.concatenate(labels)
    metrics = {}
    for i, coord in enumerate(['x', 'y', 'z']):
        metrics[coord] = {
            'mae': float(mean_absolute_error(l[:, i], p[:, i])),
            'rmse': float(np.sqrt(mean_squared_error(l[:, i], p[:, i]))),
            'r2': float(r2_score(l[:, i], p[:, i])),
        }
    dist = np.sqrt(((p - l)**2).sum(axis=1))
    metrics['3d_dist'] = {'mae': float(dist.mean()), 'median': float(np.median(dist)), 'p90': float(np.percentile(dist, 90))}
    print(f'\n=== {name} Test Results ===')
    for c in ['x','y','z']:
        print(f'  {c}: MAE={metrics[c]["mae"]:.4f}m  RMSE={metrics[c]["rmse"]:.4f}m  R2={metrics[c]["r2"]:.4f}')
    print(f'  3D: mean={metrics["3d_dist"]["mae"]:.4f}m  median={metrics["3d_dist"]["median"]:.4f}m  p90={metrics["3d_dist"]["p90"]:.4f}m')
    return p, l, metrics

print('Training utilities ready')

---
## Experiment 1: Multi-Scale CNN
Parallel conv branches with different kernel sizes capture both fine and coarse subcarrier patterns.

In [ ]:
class MultiScaleCNN(nn.Module):
    def __init__(self, n_sub=128, dropout=0.3):
        super().__init__()
        # Three parallel branches with different receptive fields
        self.branch_fine = nn.Sequential(
            nn.Conv1d(2, 64, 3, padding=1), nn.BatchNorm1d(64), nn.ReLU(),
            nn.Conv1d(64, 64, 3, padding=1), nn.BatchNorm1d(64), nn.ReLU(),
        )
        self.branch_mid = nn.Sequential(
            nn.Conv1d(2, 64, 7, padding=3), nn.BatchNorm1d(64), nn.ReLU(),
            nn.Conv1d(64, 64, 7, padding=3), nn.BatchNorm1d(64), nn.ReLU(),
        )
        self.branch_wide = nn.Sequential(
            nn.Conv1d(2, 64, 15, padding=7), nn.BatchNorm1d(64), nn.ReLU(),
            nn.Conv1d(64, 64, 15, padding=7), nn.BatchNorm1d(64), nn.ReLU(),
        )
        # Merge: 64*3 = 192 channels
        self.merge = nn.Sequential(
            nn.Conv1d(192, 256, 3, padding=1), nn.BatchNorm1d(256), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(256, 256, 3, padding=1), nn.BatchNorm1d(256), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(256, 512, 3, padding=1), nn.BatchNorm1d(512), nn.ReLU(), nn.MaxPool1d(2),
            nn.Conv1d(512, 512, 3, padding=1), nn.BatchNorm1d(512), nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.regressor = nn.Sequential(
            nn.Flatten(), nn.Linear(512, 256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, 64), nn.ReLU(), nn.Dropout(dropout), nn.Linear(64, 3),
        )

    def forward(self, x):
        f = self.branch_fine(x)
        m = self.branch_mid(x)
        w = self.branch_wide(x)
        x = torch.cat([f, m, w], dim=1)
        x = self.merge(x)
        return self.regressor(x)

model1 = MultiScaleCNN().to(device)
print(f'MultiScale params: {sum(p.numel() for p in model1.parameters()):,}')
hist1, best1 = train_model(model1, 'multiscale', epochs=120)

In [ ]:
model1.load_state_dict(torch.load(f'{SAVE_DIR}/best_multiscale.pt', weights_only=True))
preds1, labels1, metrics1 = eval_model(model1, 'Multi-Scale CNN')

## Experiment 2: Transformer Encoder
Self-attention over subcarrier positions captures global dependencies.

In [ ]:
class CSI_Transformer(nn.Module):
    def __init__(self, n_sub=128, d_model=64, nhead=4, n_layers=4, dropout=0.3):
        super().__init__()
        self.input_proj = nn.Linear(2, d_model)
        self.pos_embed = nn.Parameter(torch.randn(1, n_sub, d_model) * 0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model*4,
            dropout=dropout, batch_first=True, activation='gelu',
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.pool = nn.Sequential(
            nn.AdaptiveAvgPool1d(1), nn.Flatten(),
        )
        self.regressor = nn.Sequential(
            nn.Linear(d_model, 128), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(128, 64), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(64, 3),
        )

    def forward(self, x):
        # x: (B, 2, 128) -> (B, 128, 2)
        x = x.permute(0, 2, 1)
        x = self.input_proj(x) + self.pos_embed
        x = self.encoder(x)
        # (B, 128, d_model) -> pool over sequence
        x = x.permute(0, 2, 1)  # (B, d_model, 128)
        x = self.pool(x)
        return self.regressor(x)

model2 = CSI_Transformer(n_sub=N_SUB, d_model=64, nhead=4, n_layers=4).to(device)
print(f'Transformer params: {sum(p.numel() for p in model2.parameters()):,}')
hist2, best2 = train_model(model2, 'transformer', epochs=120, lr=5e-4, max_lr=2e-3)

In [ ]:
model2.load_state_dict(torch.load(f'{SAVE_DIR}/best_transformer.pt', weights_only=True))
preds2, labels2, metrics2 = eval_model(model2, 'Transformer')

## Experiment 3: Deep ResNet + SE Attention
Squeeze-and-Excitation blocks let the network learn which channels matter most.

In [ ]:
class SEBlock(nn.Module):
    """Squeeze-and-Excitation channel attention"""
    def __init__(self, channels, reduction=4):
        super().__init__()
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool1d(1), nn.Flatten(),
            nn.Linear(channels, channels // reduction), nn.ReLU(),
            nn.Linear(channels // reduction, channels), nn.Sigmoid(),
        )
    def forward(self, x):
        w = self.fc(x).unsqueeze(-1)
        return x * w

class SEResBlock(nn.Module):
    def __init__(self, channels, kernel_size=3):
        super().__init__()
        self.conv1 = nn.Conv1d(channels, channels, kernel_size, padding=kernel_size//2)
        self.bn1 = nn.BatchNorm1d(channels)
        self.conv2 = nn.Conv1d(channels, channels, kernel_size, padding=kernel_size//2)
        self.bn2 = nn.BatchNorm1d(channels)
        self.se = SEBlock(channels)
    def forward(self, x):
        residual = x
        x = torch.relu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        x = self.se(x)
        return torch.relu(x + residual)

class DeepSEResNet(nn.Module):
    def __init__(self, n_sub=128, dropout=0.3):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(2, 64, 7, padding=3), nn.BatchNorm1d(64), nn.ReLU(),
        )
        self.layer1 = nn.Sequential(SEResBlock(64), SEResBlock(64), SEResBlock(64), nn.MaxPool1d(2))
        self.expand1 = nn.Sequential(nn.Conv1d(64, 128, 1), nn.BatchNorm1d(128), nn.ReLU())
        self.layer2 = nn.Sequential(SEResBlock(128), SEResBlock(128), SEResBlock(128), nn.MaxPool1d(2))
        self.expand2 = nn.Sequential(nn.Conv1d(128, 256, 1), nn.BatchNorm1d(256), nn.ReLU())
        self.layer3 = nn.Sequential(SEResBlock(256), SEResBlock(256), SEResBlock(256), nn.MaxPool1d(2))
        self.expand3 = nn.Sequential(nn.Conv1d(256, 512, 1), nn.BatchNorm1d(512), nn.ReLU())
        self.layer4 = nn.Sequential(SEResBlock(512), SEResBlock(512))
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.regressor = nn.Sequential(
            nn.Flatten(), nn.Linear(512, 256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, 64), nn.ReLU(), nn.Dropout(dropout), nn.Linear(64, 3),
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.expand1(self.layer1(x))
        x = self.expand2(self.layer2(x))
        x = self.expand3(self.layer3(x))
        x = self.pool(self.layer4(x))
        return self.regressor(x)

model3 = DeepSEResNet().to(device)
print(f'DeepSEResNet params: {sum(p.numel() for p in model3.parameters()):,}')
hist3, best3 = train_model(model3, 'se_resnet', epochs=120)

In [ ]:
model3.load_state_dict(torch.load(f'{SAVE_DIR}/best_se_resnet.pt', weights_only=True))
preds3, labels3, metrics3 = eval_model(model3, 'Deep SE-ResNet')

## Experiment 4: Ensemble (average of all three)

In [ ]:
# Ensemble: weighted average by inverse 3D MAE
w1 = 1.0 / metrics1['3d_dist']['mae']
w2 = 1.0 / metrics2['3d_dist']['mae']
w3 = 1.0 / metrics3['3d_dist']['mae']
wt = w1 + w2 + w3

ens_preds = (w1 * preds1 + w2 * preds2 + w3 * preds3) / wt
Y_test = labels1  # same for all

print(f'\n=== Ensemble Test Results ===')
print(f'Weights: MultiScale={w1/wt:.3f}, Transformer={w2/wt:.3f}, SE-ResNet={w3/wt:.3f}')
ens_metrics = {}
for i, coord in enumerate(['x', 'y', 'z']):
    mae = float(mean_absolute_error(Y_test[:, i], ens_preds[:, i]))
    rmse = float(np.sqrt(mean_squared_error(Y_test[:, i], ens_preds[:, i])))
    r2 = float(r2_score(Y_test[:, i], ens_preds[:, i]))
    ens_metrics[coord] = {'mae': mae, 'rmse': rmse, 'r2': r2}
    print(f'  {coord}: MAE={mae:.4f}m  RMSE={rmse:.4f}m  R2={r2:.4f}')

dist_ens = np.sqrt(((ens_preds - Y_test)**2).sum(axis=1))
ens_metrics['3d_dist'] = {'mae': float(dist_ens.mean()), 'median': float(np.median(dist_ens)), 'p90': float(np.percentile(dist_ens, 90))}
print(f'  3D: mean={dist_ens.mean():.4f}m  median={np.median(dist_ens):.4f}m  p90={np.percentile(dist_ens, 90):.4f}m')

## Final Comparison

In [ ]:
all_models = {
    'v1 ResNet (baseline)': {'3d_dist': {'mae': 2.837}},
    'v1 LightGBM (baseline)': {'3d_dist': {'mae': 3.212}},
    'Multi-Scale CNN': metrics1,
    'Transformer': metrics2,
    'Deep SE-ResNet': metrics3,
    'Ensemble': ens_metrics,
}

print(f'\n{"="*70}')
print(f'{"Model":<25} {"x MAE":>8} {"y MAE":>8} {"z MAE":>8} {"3D dist":>10} {"vs base":>10}')
print(f'{"="*70}')
for name, m in all_models.items():
    d3 = m['3d_dist']['mae']
    diff = d3 - 2.837
    sign = '+' if diff >= 0 else ''
    if 'x' in m:
        print(f'{name:<25} {m["x"]["mae"]:>8.3f} {m["y"]["mae"]:>8.3f} {m["z"]["mae"]:>8.3f} {d3:>10.3f} {sign}{diff:>9.3f}')
    else:
        print(f'{name:<25} {"—":>8} {"—":>8} {"—":>8} {d3:>10.3f} {sign}{diff:>9.3f}')
print(f'{"="*70}')

# Bar chart
names = list(all_models.keys())
vals = [all_models[n]['3d_dist']['mae'] for n in names]
colors = ['gray', 'gray', 'steelblue', 'steelblue', 'steelblue', 'green']
plt.figure(figsize=(10, 5))
plt.barh(names, vals, color=colors)
plt.xlabel('3D Distance Error (m)')
plt.title('Localization Model Comparison')
for i, v in enumerate(vals):
    plt.text(v + 0.02, i, f'{v:.3f}m', va='center')
plt.tight_layout(); plt.show()

# Training curves
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, h, name in zip(axes, [hist1, hist2, hist3], ['Multi-Scale', 'Transformer', 'SE-ResNet']):
    ax.plot(h['val_mae'], label='Val 3D MAE')
    ax.set_xlabel('Epoch'); ax.set_ylabel('3D MAE (m)'); ax.set_title(name)
    ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# Save everything
np.savez(f'{SAVE_DIR}/loc_v2_norm_stats.npz', X_mean=X_mean, X_std=X_std, Y_mean=Y_mean, Y_std=Y_std)

results = {
    'task': 'localization_v2',
    'models': {
        name: {k: {kk: round(float(vv), 6) for kk, vv in v.items()} for k, v in m.items() if isinstance(v, dict)}
        for name, m in all_models.items()
    },
    'best_model': min(all_models, key=lambda n: all_models[n]['3d_dist']['mae']),
    'best_3d_mae': round(float(min(m['3d_dist']['mae'] for m in all_models.values())), 6),
    'histories': {
        'multiscale': {k: [float(v) for v in vals] for k, vals in hist1.items()},
        'transformer': {k: [float(v) for v in vals] for k, vals in hist2.items()},
        'se_resnet': {k: [float(v) for v in vals] for k, vals in hist3.items()},
    },
}
with open(f'{SAVE_DIR}/loc_v2_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print(f'\nSaved to: {SAVE_DIR}/')
for fname in sorted(os.listdir(SAVE_DIR)):
    if 'loc' in fname or 'multiscale' in fname or 'transformer' in fname or 'se_resnet' in fname:
        size = os.path.getsize(os.path.join(SAVE_DIR, fname))
        print(f'  {fname} ({size/1e6:.1f} MB)' if size > 1e6 else f'  {fname} ({size/1e3:.0f} KB)')

## Summary — Copy this!

In [ ]:
print('=== COPY EVERYTHING BELOW ===')
print(json.dumps(results, indent=2))